### Aula 1 - Tratamento, exploração e visualização de dados

In [1]:
# Importar a biblioteca
import pandas as pd
from io import StringIO
import re

import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
# Ler o arquivo e gerar um data frame
caminho_arquivo = r"C:\Users\ricar\OneDrive\Cursos\Pós Graduação\Data Analytics - FIAP\02 - Fase 1 - Data Analysis and Exploration\03 - Visualização de dados\Bases\estimativa_dou_2020.xls"                                             
ibge_estimativa = pd.read_excel(caminho_arquivo)
ibge_estimativa.head()

,ESTIMATIVAS DA POPULAÇÃO RESIDENTE NO BRASIL E UNIDADES DA FEDERAÇÃO COM DATA DE REFERÊNCIA EM 1º DE JULHO DE 2020,Unnamed: 1,Unnamed: 2
0,BRASIL E UNIDADES DA FEDERAÇÃO,NaN,POPULAÇÃO ESTIMADA
1,Brasil,NaN,211755692
2,Região Norte,NaN,18672591
3,Rondônia,NaN,1796460
4,Acre,NaN,894470


In [3]:
dados_da_populacao = """Posição	Unidade federativa	População	% da pop. total	País comparável
(habitantes)

1	 São Paulo	46 649 132	21,9%	Flag of Spain.svg Espanha (46 439 864)
2	 Minas Gerais	21 411 923	10,1%	 Sri Lanka (20 675 000)
3	 Rio de Janeiro	17 463 349	8,2%	 Países Baixos (16 922 900)
4	Bahia Bahia	14 985 284	7,1%	 Chade (14 037 000)
5	 Paraná	11 597 484	5,4%	 Bolívia (11 410 651)
6	 Rio Grande do Sul	11 466 630	5,4%	 Bélgica (11 250 659)
7	 Pernambuco	9 674 793	4,5%	 Bielorrússia (9 485 300)
8	 Ceará	9 240 580	4,3%	 Emirados Árabes Unidos (9 157 000)
9	Pará Pará	8 777 124	4,1%	 Áustria (8 602 112)
10	 Santa Catarina	7 338 473	3,4%	 Sérvia (7 114 393)
11	 Goiás	7 206 589	3,4%	 Paraguai (7 003 406)
12	 Maranhão	7 153 262	3,4%	 Paraguai (7 003 406)
13	 Amazonas	4 269 995	2,0%	 Líbano (4 168 000)
14	 Espírito Santo	4 108 508	1,9%	 Líbano (4 168 000)
15	 Paraíba	4 059 905	1,9%	 Líbano (4 168 000)
16	 Mato Grosso	3 567 234	1,7%	 Uruguai (3 415 866)
17	 Rio Grande do Norte	3 560 903	1,7%	 Uruguai (3 415 866)
18	 Alagoas	3 365 351	1,6%	 Uruguai (3 415 866)
19	 Piauí	3 289 290	1,6%	 Kuwait (3 268 431)
20	 Distrito Federal	3 094 325	1,4%	 Lituânia (2 900 787)
21	 Mato Grosso do Sul	2 839 188	1,3%	 Jamaica (2 717 991)
22	 Sergipe	2 338 474	1,1%	 Namíbia (2 280 700)
23	 Rondônia	1 815 278	0,8%	 Gabão (1 725 000)
24	 Tocantins	1 607 363	0,7%	 Bahrein (1 359 800)
25	 Acre	906 876	0,4%	 Fiji (859 178)
26	 Amapá	877 613	0,4%	 Fiji (859 178)
27	 Roraima	652 713	0,3%	 Luxemburgo (562 958)"""

# fonte https://pt.wikipedia.org/wiki/Lista_de_unidades_federativas_do_Brasil_por_popula%C3%A7%C3%A3o#cite_note-IBGE_POP-1
# fonte indireta IBGE

In [4]:
dados_da_populacao_io = StringIO(dados_da_populacao)

populacao = pd.read_csv(dados_da_populacao_io, sep="\t")
populacao = populacao.dropna()
populacao.head()

,Posição,Unidade federativa,População,% da pop. total,País comparável
1,1,São Paulo,46 649 132,"21,9%",Flag of Spain.svg Espanha (46 439 864)
2,2,Minas Gerais,21 411 923,"10,1%",Sri Lanka (20 675 000)
3,3,Rio de Janeiro,17 463 349,"8,2%",Países Baixos (16 922 900)
4,4,Bahia Bahia,14 985 284,"7,1%",Chade (14 037 000)
5,5,Paraná,11 597 484,"5,4%",Bolívia (11 410 651)


##### Desafio 1 - Tratar o Excel

In [5]:
# Ler o arquivo e gerar um data frame
caminho_arquivo = r"C:\Users\ricar\OneDrive\Cursos\Pós Graduação\Data Analytics - FIAP\02 - Fase 1 - Data Analysis and Exploration\03 - Visualização de dados\Bases\estimativa_dou_2020.xls"                                             

# Gerar o data frame a partir de um arquivo Excel
ibge_estimativa_tratado = pd.read_excel(caminho_arquivo, skiprows=3, skipfooter=7, engine='xlrd')

# Mudar nome das colunas
ibge_estimativa_tratado.columns = ["Unidade Federativa","Remover","Estimativa População"]

# Remover coluna que não é necessaria
ibge_estimativa_tratado = ibge_estimativa_tratado.drop(columns=["Remover"])

# Remover todas as linhas que contêm a palavra "Região" na coluna "Unidade Federativa"
ibge_estimativa_tratado = ibge_estimativa_tratado[~ibge_estimativa_tratado["Unidade Federativa"].str.contains("Região", na=False)]

# Remover parênteses e tudo que está dentro deles apenas se existirem
ibge_estimativa_tratado['Estimativa População'] = ibge_estimativa_tratado['Estimativa População'].apply(
    lambda x: re.sub(r'\(.*?\)', '', str(x)) if '(' in str(x) else x)

# Remover pontos dos valores na coluna "Estimativa População"
ibge_estimativa_tratado['Estimativa População'] = ibge_estimativa_tratado['Estimativa População'].apply(
    lambda x: str(x).replace('.', '') if isinstance(x, str) else x)

# Converter a coluna "Estimativa População" para números
ibge_estimativa_tratado['Estimativa População'] = pd.to_numeric(ibge_estimativa_tratado['Estimativa População'], errors='coerce')

# Ver os dados tratados
ibge_estimativa_tratado.head()

,Unidade Federativa,Estimativa População
0,Rondônia,1796460
1,Acre,894470
2,Amazonas,4207714
3,Roraima,631181
4,Pará,8690745


#### Desafio 2 - Extrair tabela do Wikipedia

In [6]:

# URL da página da Wikipedia
url = "https://pt.wikipedia.org/wiki/Lista_de_unidades_federativas_do_Brasil_por_popula%C3%A7%C3%A3o"

# Fazer a requisição HTTP para obter o conteúdo da página
response = requests.get(url)
html_content = response.content

# Analisar o HTML com BeautifulSoup
soup = BeautifulSoup(html_content, 'html.parser')

# Localizar a tabela desejada (primeira tabela relevante)
tabela = soup.select_one("table.wikitable")

# Extrair os cabeçalhos da tabela
cabecalhos = [th.text.strip() for th in tabela.select("thead tr th")]

# Extrair os dados das linhas da tabela
linhas = []
for tr in tabela.select("tbody tr"):
    colunas = [td.text.strip() for td in tr.select("td")]
    if colunas:  # Ignorar linhas vazias
        linhas.append(colunas)

cabecalhos = [f"Coluna {i+1}" for i in range(len(linhas[0]))]

# Criar um DataFrame com os dados extraídos
tabela_wiki = pd.DataFrame(linhas, columns=cabecalhos)

# Manter apenas as colunas necessarias
tabela_wiki = tabela_wiki[["Coluna 1","Coluna 2"]]

# Mudar nome das colunas
tabela_wiki.columns = ["Unidade Federativa","População"]

# Remover espaços não quebráveis (\xa0) e outros caracteres indesejados da coluna 'População'
tabela_wiki['População'] = tabela_wiki['População'].str.replace(r'\xa0', '', regex=True)
tabela_wiki['População'] = tabela_wiki['População'].str.replace(r'\s+', '', regex=True)

tabela_wiki['População'] = pd.to_numeric(tabela_wiki['População'], errors='coerce')

# Exibir as primeiras linhas do DataFrame
tabela_wiki.head()

,Unidade Federativa,População
0,São Paulo,45973194
1,Minas Gerais,21322691
2,Rio de Janeiro,17219679
3,Bahia,14850513
4,Paraná,11824665


### Aula 2 - Proporcionalidade e Seaborn

In [7]:
# Renomear as colunas
populacao.columns = ["posicao","uf","populacao","porcentagem","pais_comparavel"]

# Tirar o espaço da coluna populacao
populacao["populacao"] = populacao["populacao"].str.replace(" ","").astype(int)

populacao = populacao[["uf","populacao"]]

populacao.head()

,uf,populacao
1,São Paulo,46649132
2,Minas Gerais,21411923
3,Rio de Janeiro,17463349
4,Bahia Bahia,14985284
5,Paraná,11597484


### Aula 3 - Ticks, escalas e formação de imagem

### Aula 4 - Trabalhando Datetime e Melt

### Aula 5 - Manipulando datas e gerando novas análises

### Aula 6 - Agrupando Dados e Analisando por Categoria